In [ ]:
!uv pip install torch accelerate transformers pytorch-lightning tslearn --quiet

In [ ]:
# from google.colab import drive
# drive.mount('content/')
# _path = 'content/My Drive/Colab Notebooks/GP Datasets/25_countries_main.csv'

In [ ]:
import pandas as pd
import numpy as np

_path = '/kaggle/input/datasets/sohumgokhale/global-climate-health-impact-tracker-2015-2025/global_climate_health_impact_tracker_2015_2025.csv'
data_df = pd.read_csv(_path)

# =============================================================================
# CONSTANTS
# =============================================================================
FEATURES = [
    'temp_squared', 'pm25_ugm3', 'heat_wave_days', 'month_sin',
    'temp_precip_interaction',
    # 'aqi_pm',
    'temp_anomaly_celsius',
    'pollution_vulnerability', 'flood_indicator', 'healthcare_access_index',
    'distance_to_equator', 'pm25_change_rate', 'pm25_ugm3_lag_1w', 'month_cos',
    'temp_change_rate', 'temperature_celsius', 'gdp_per_capita_usd',
    'quarter', 'spatial_lag_pm25', 'is_northern',
    'is_tropical', 'spatial_lag_temp', 'food_security_index',
]

TARGETS = [
    'respiratory_disease_rate',
    'cardio_mortality_rate',
    'vector_disease_risk_score',
    'waterborne_disease_incidents',
    'heat_related_admissions',
]

TARGET_LABELS = {t: t.replace("_", " ").title() for t in TARGETS}

TARGET_COLORS = {
    "respiratory_disease_rate":     "#4fc3f7",
    "cardio_mortality_rate":        "#81c784",
    "vector_disease_risk_score":    "#ffb74d",
    "waterborne_disease_incidents": "#f06292",
    "heat_related_admissions":      "#ce93d8",
}

SEQ_LEN = 52
HORIZON_LEN = 13
NUM_FEATURES = len(FEATURES)
NUM_TARGETS = len(TARGETS)
ALL_COLS = FEATURES + TARGETS
COUNTRIES = data_df['country_code'].unique()
NUM_COUNTRIES = len(COUNTRIES)


# =============================================================================
# FEATURE ENGINEERING
# =============================================================================
df = data_df.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country_code', 'date']).reset_index(drop=True)

df['temp_squared'] = df['temperature_celsius'] ** 2
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['quarter'] = ((df['month'] - 1) // 3 + 1).astype(float)
df['temp_precip_interaction'] = df['temperature_celsius'] * df['precipitation_mm']
df['pollution_vulnerability'] = df['pm25_ugm3'] / (df['healthcare_access_index'] + 1e-6)
df['distance_to_equator'] = df['latitude'].abs()
df['is_northern'] = (df['latitude'] > 0).astype(float)
df['is_tropical'] = (df['latitude'].abs() < 23.5).astype(float)

df['pm25_change_rate'] = df.groupby('country_code')['pm25_ugm3'].diff()
df['temp_change_rate'] = df.groupby('country_code')['temperature_celsius'].diff()
df['pm25_ugm3_lag_1w'] = df.groupby('country_code')['pm25_ugm3'].shift(1)

for col, new_col in [('pm25_ugm3', 'spatial_lag_pm25'), ('temperature_celsius', 'spatial_lag_temp')]:
    week_sum = df.groupby('date')[col].transform('sum')
    week_count = df.groupby('date')[col].transform('count')
    df[new_col] = (week_sum - df[col]) / (week_count - 1)

for col in ['pm25_change_rate', 'temp_change_rate', 'pm25_ugm3_lag_1w']:
    df[col] = df.groupby('country_code')[col].bfill()

# Integer time index per country (used for splitting)
df['time_idx'] = df.groupby('country_code').cumcount()

# Validate
assert set(FEATURES).issubset(df.columns), f"Missing: {set(FEATURES) - set(df.columns)}"
assert set(TARGETS).issubset(df.columns), f"Missing: {set(TARGETS) - set(df.columns)}"
assert df[ALL_COLS].isna().sum().sum() == 0, "NaN found in features/targets"

df_engineered = df

print(f"Shape: {df_engineered.shape[0]} rows × {df_engineered.shape[1]} cols")
print(f"Features: {NUM_FEATURES} | Targets: {NUM_TARGETS}")
print(f"Countries: {df['country_code'].nunique()} | Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"time_idx range: 0 → {df['time_idx'].max()}")

In [ ]:
train_ratio, val_ratio = 0.80, 0.90
max_time_idx = df_engineered['time_idx'].max()
train_cutoff = int(max_time_idx * train_ratio)
val_cutoff = int(max_time_idx * val_ratio)

train_df = df_engineered[df_engineered['time_idx'] <= train_cutoff].copy()
val_df = df_engineered[(df_engineered['time_idx'] > train_cutoff) &
                        (df_engineered['time_idx'] <= val_cutoff)].copy()
test_df = df_engineered[df_engineered['time_idx'] > val_cutoff].copy()

print(f"Train: {len(train_df):,} rows (time_idx 0-{train_cutoff})")
print(f"Val:   {len(val_df):,} rows  (time_idx {train_cutoff+1}-{val_cutoff})")
print(f"Test:  {len(test_df):,} rows  (time_idx {val_cutoff+1}-{max_time_idx})")
print(f"\ntrain_cutoff={train_cutoff} | val_cutoff={val_cutoff}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
import joblib

scaler = RobustScaler()
scaler.fit(train_df[ALL_COLS].values)

TARGET_INDICES = [ALL_COLS.index(t) for t in TARGETS]

train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

for split_df in [train_scaled, val_scaled, test_scaled]:
    split_df[ALL_COLS] = scaler.transform(split_df[ALL_COLS].values)

joblib.dump(scaler, "scaler.save")

class DecoderDataset(Dataset):
    """
    Sliding windows per country:
        input_seq  : X_y[k : k + SEQ_LEN]       → [SEQ_LEN, 28]  (features + targets)
        target_seq : y[k+1 : k + SEQ_LEN + 1]   → [SEQ_LEN, 5]   (targets only, shifted +1)

    min_target_idx: if set, only include windows whose last target step
                    (time_idx at position k + SEQ_LEN) > min_target_idx.
                    This allows val/test sets to draw history from earlier splits
                    while ensuring evaluation targets fall in the right period.
    """
    def __init__(self, df, seq_len, features, targets, min_target_idx=None):
        self.seq_len = seq_len
        all_cols = features + targets
        target_cols = targets
        self.samples = []

        for _, group in df.groupby('country_code'):
            group = group.sort_values('time_idx')
            data = group[all_cols].values.astype(np.float32)
            tgt  = group[target_cols].values.astype(np.float32)
            tidx = group['time_idx'].values
            T = len(data)
            for k in range(T - seq_len):
                if min_target_idx is not None and tidx[k + seq_len] <= min_target_idx:
                    continue
                x = data[k : k + seq_len]
                y = tgt[k + 1 : k + seq_len + 1]
                self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)

BATCH_SIZE = 256

train_dataset = DecoderDataset(train_scaled, SEQ_LEN, FEATURES, TARGETS)

val_combined = pd.concat([train_scaled, val_scaled]).sort_values(['country_code', 'time_idx'])
test_combined = pd.concat([train_scaled, val_scaled, test_scaled]).sort_values(['country_code', 'time_idx'])

val_dataset  = DecoderDataset(val_combined,  SEQ_LEN, FEATURES, TARGETS, min_target_idx=train_cutoff)
test_dataset = DecoderDataset(test_combined, SEQ_LEN, FEATURES, TARGETS, min_target_idx=val_cutoff)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=3)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=3)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=3)

print(f"Train samples: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")
print(f"Batch size: {BATCH_SIZE} | Train batches/epoch: {len(train_loader)}")
print(f"Input shape per sample: [{SEQ_LEN}, {len(ALL_COLS)}] | Target shape: [{SEQ_LEN}, {NUM_TARGETS}]")

In [ ]:
import torch.nn as nn
import pytorch_lightning as pl
from transformers import GPT2Config, GPT2Model
from tslearn.metrics import SoftDTWLossPyTorch
import torch.nn.functional as F

class AsymmetricMSELoss(nn.Module):
    def __init__(self, alpha=0.8):
        super().__init__()
        self.alpha = alpha

    def forward(self, predicted, target):
        residual = target - predicted
        # Apply larger weight to positive residuals (target > predicted)
        weight = torch.where(residual > 0, self.alpha, 1 - self.alpha)
        loss = weight * (residual ** 2)
        return loss.mean()

class LogCoshLoss(nn.Module):
    def __init__(self, eps=1e-12):
        super(LogCoshLoss, self).__init__()
        self.eps = eps

    def forward(self, y_t, y_p):
        ey = y_p - y_t
        return torch.mean(torch.log(torch.cosh(ey + self.eps)))

class EnhancedPeakLoss(nn.Module):
    def __init__(self, threshold=0.95):
        super(EnhancedPeakLoss, self).__init__()
        self.threshold = threshold

    def forward(self, y_pred, y_true):
        error = torch.abs(y_pred - y_true)
        threshold_val = torch.quantile(error, self.threshold)

        penalty = torch.where(error > threshold_val, 2.0, 1.0)

        loss = penalty * F.mse_loss(y_pred, y_true, reduction='none')
        return loss.mean()

class RobustSignalDecayLoss(nn.Module):
    def __init__(self, a=1.0, lambda_val=1.0):
        super(RobustSignalDecayLoss, self).__init__()
        self.a = a
        self.lambda_val = lambda_val

    def forward(self, predicted, target):
        residuals = target - predicted
        u_sq = residuals**2

        inner = 1.0 + self.a * u_sq

        decay = torch.pow(inner, -self.a) * torch.exp(-(self.a * u_sq))

        loss = self.lambda_val * (1 - decay)

        return torch.mean(loss)

class GPT2Forecaster(pl.LightningModule):
    def __init__(self, n_features=len(ALL_COLS), n_targets=NUM_TARGETS,
                 seq_len=SEQ_LEN, horizon_len=HORIZON_LEN, d_model=96, n_layer=4, n_head=4,
                 dropout=0.15, lr=1e-3, weight_decay=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.input_proj = nn.Sequential(
            nn.Linear(n_features, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model),
            nn.LayerNorm(d_model),
        )
        # Component 2 — GPT2 Backbone (causal mask built-in)
        self.backbone = GPT2Model(GPT2Config(
            n_embd=d_model,
            n_layer=n_layer,
            n_head=n_head,
            n_positions=seq_len + horizon_len,
            resid_pdrop=dropout,
            attn_pdrop=dropout,
            embd_pdrop=dropout,
            use_cache=False,  # off during training, on during inference
            vocab_size=1,     # unused, but required by config
        ))

        # Component 3 — Output Head
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_targets),
        )

        self.loss_fn = nn.HuberLoss()
        # self.loss_fn = SoftDTWLossPyTorch(gamma=0.05)
        # self.loss_fn = nn.MSELoss(reduction='mean')
        # self.loss_fn = nn.L1Loss()
        # self.loss_fn = RobustSignalDecayLoss()
        # self.loss_fn = EnhancedPeakLoss()
        # self.loss_fn = LogCoshLoss()
        # self.loss_fn = AsymmetricMSELoss()

    def forward(self, x, use_cache=False, past_key_values=None, position_ids=None):
        """
        x: [B, T, n_features]  (features + targets concatenated)
        Returns: preds [B, T, n_targets], past_key_values (if use_cache)
        """
        h = self.input_proj(x)  # [B, T, d_model]
        out = self.backbone(
            inputs_embeds=h,
            use_cache=use_cache,
            past_key_values=past_key_values,
            position_ids=position_ids,
        )
        preds = self.output_head(out.last_hidden_state)  # [B, T, n_targets]
        if use_cache:
            return preds, out.past_key_values
        return preds

    def training_step(self, batch, batch_idx):
        x, y = batch  # x: [B, SEQ_LEN, 28], y: [B, SEQ_LEN, 5]
        preds = self.forward(x)
        loss = self.loss_fn(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self.forward(x)
        loss = self.loss_fn(preds, y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=5, factor=0.5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }


model = GPT2Forecaster()
model

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GPT2Forecaster — Total params: {total_params:,} | Trainable: {trainable_params:,}")
print(f"Sequence length: {SEQ_LEN} | Horizon: {HORIZON_LEN} | Targets: {NUM_TARGETS}")
print(f"Loss: HuberLoss (next-token prediction)")

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

def train_model(model, train_loader, val_loader, max_epochs=100, patience=25):
    """Train GPT2Forecaster using Lightning Trainer with early stopping."""
    early_stop = EarlyStopping(monitor="val_loss", patience=patience, mode="min")
    lr_monitor = LearningRateMonitor(logging_interval="epoch")

    trainer = pl.Trainer(
        default_root_dir='/kaggle/working/checkpoints/',
        max_epochs=max_epochs,
        accelerator="auto",
        gradient_clip_val=0.1,
        callbacks=[early_stop, lr_monitor],
        enable_progress_bar=True,
        log_every_n_steps=5,
    )
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    best_model_path = trainer.checkpoint_callback.best_model_path
    best_model = GPT2Forecaster.load_from_checkpoint(best_model_path)
    print(f"\nBest model loaded from: {best_model_path}")
    return best_model, trainer


# =============================================
# >>> UNCOMMENT BELOW TO START TRAINING    <<<
# >>> (Long-running: ~5-15 min on GPU)     <<<
# =============================================
best_model, trainer = train_model(model, train_loader, val_loader, max_epochs=300, patience=35)

In [ ]:
best_model = GPT2Forecaster.load_from_checkpoint( # HuberLoss val_loss: 0.201
    '/kaggle/working/checkpoints/lightning_logs/version_0/checkpoints/epoch=181-step=7098.ckpt'
)
# best_model = GPT2Forecaster.load_from_checkpoint( # LogCoshLoss
#     '/kaggle/working/checkpoints/lightning_logs/version_18/checkpoints/epoch=133-step=5226.ckpt'
# )
# best_model = GPT2Forecaster.load_from_checkpoint('/kaggle/working/checkpoints/lightning_logs/version_6/checkpoints/epoch=19-step=780.ckpt')
# best_model = GPT2Forecaster.load_from_checkpoint('/kaggle/working/checkpoints/lightning_logs/version_7/checkpoints/epoch=49-step=1950.ckpt')
# best_model = GPT2Forecaster.load_from_checkpoint('/kaggle/working/checkpoints/lightning_logs/version_8/checkpoints/epoch=49-step=1950.ckpt')
# best_model = GPT2Forecaster.load_from_checkpoint('/kaggle/working/checkpoints/lightning_logs/version_13/checkpoints/epoch=49-step=1950.ckpt')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def autoregressive_forecast(model, context, horizon, n_features, n_targets, eval_mode=True):
    """
    Autoregressive rollout for HORIZON_LEN steps using KV cache.
    context : [B, SEQ_LEN, n_features + n_targets]  scaled
    Returns : [B, horizon, n_targets]  scaled
    """
    if eval_mode:
        model.eval()
    device = next(model.parameters()).device
    context = context.to(device)

    preds_list = []
    past_kv = None
    next_pred = None

    with torch.no_grad():
        for step in range(horizon):
            if step == 0:
                out, past_kv = model.forward(context, use_cache=True)
                next_pred    = out[:, -1:, :] # [B, 1, n_targets]
            else:
                # Carry forward last known features, replace targets with last prediction
                last_token = context[:, -1:, :].clone() # [B, 1, n_features + n_targets]
                last_token[:, :, -n_targets:] = next_pred

                cache_len = past_kv.get_seq_length() # increments automatically each step
                pos_ids   = torch.full((context.shape[0], 1), cache_len, dtype=torch.long, device=device)

                out, past_kv = model.forward(last_token, use_cache=True, past_key_values=past_kv, position_ids=pos_ids)
                next_pred = out[:, -1:, :] # [B, 1, n_targets]

            preds_list.append(next_pred.cpu())

    return torch.cat(preds_list, dim=1) # [B, horizon, n_targets]


def mc_dropout_forecast(model, context, horizon, n_features, n_targets, n_samples=20):
    """
    MC Dropout: run autoregressive forecast N times with dropout enabled.
    Returns: mean [B, H, T], std [B, H, T] in scaled space.
    """
    model.train()  # enable dropout
    all_preds = []
    for _ in range(n_samples):
        with torch.no_grad():
            p = autoregressive_forecast(model, context, horizon, n_features, n_targets, eval_mode=False)
            all_preds.append(p.cpu())
    stacked = torch.stack(all_preds)  # [N, B, H, T]
    model.eval()
    return stacked.mean(dim=0), stacked.std(dim=0)

def inverse_scale_targets(arr, scaler, target_indices):
    """Inverse-transform target columns only, using the full scaler."""
    dummy = np.zeros((arr.shape[0], len(scaler.center_)))
    dummy[:, target_indices] = arr
    inv = scaler.inverse_transform(dummy)
    return inv[:, target_indices]


def evaluate_model(model, test_loader, scaler, target_indices, targets,
                   horizon, n_features, n_targets, n_mc=20):
    """Evaluate GPT2Forecaster: autoregressive rollout + metrics + plots."""
    model.eval()
    device = next(model.parameters()).device

    all_preds, all_stds, all_actuals = [], [], []

    for x_batch, y_batch in test_loader:
        # Actuals: last HORIZON positions of the shifted target
        actual_scaled = y_batch[:, -horizon:, :].numpy()  # [B, H, 5]

        # MC Dropout forecast

        mean_pred, std_pred = mc_dropout_forecast(
            model, x_batch.to(device), horizon, n_features, n_targets, n_samples=n_mc
        )
        mean_pred = mean_pred.numpy()  # [B, H, 5]
        std_pred = std_pred.numpy()

        all_preds.append(mean_pred)
        all_stds.append(std_pred)
        all_actuals.append(actual_scaled)

    # Concatenate all batches
    preds_scaled = np.concatenate(all_preds, axis=0)    # [N, H, 5]
    stds_scaled  = np.concatenate(all_stds, axis=0)
    actuals_scaled = np.concatenate(all_actuals, axis=0)

    # Inverse transform to original scale
    N, H, T = preds_scaled.shape
    preds_flat = preds_scaled.reshape(-1, T)
    actuals_flat = actuals_scaled.reshape(-1, T)
    stds_flat = stds_scaled.reshape(-1, T)

    preds_orig = inverse_scale_targets(preds_flat, scaler, target_indices).reshape(N, H, T)
    actuals_orig = inverse_scale_targets(actuals_flat, scaler, target_indices).reshape(N, H, T)
    # Scale std by the scaler's scale factor for targets
    target_scales = scaler.scale_[target_indices]
    stds_orig = stds_scaled * target_scales[None, None, :]

    # Per-target metrics
    results = []
    for i, name in enumerate(targets):
        p = preds_orig[:, :, i].flatten()
        a = actuals_orig[:, :, i].flatten()
        mae = mean_absolute_error(a, p)
        rmse = np.sqrt(mean_squared_error(a, p))
        r2 = r2_score(a, p)

        threshold = np.percentile(np.abs(a), 90)
        peak_mask = np.abs(a) > threshold
        peak_mae = mean_absolute_error(a[peak_mask], p[peak_mask]) if peak_mask.sum() > 0 else float('nan')

        da = np.diff(a.reshape(-1, horizon), axis=1)
        dp = np.diff(p.reshape(-1, horizon), axis=1)
        dir_acc = (np.sign(da) == np.sign(dp)).mean() * 100

        results.append({
            'Target': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
            'R²': round(r2, 4), 'Peak MAE': round(peak_mae, 4), 'Dir Acc %': round(dir_acc, 2),
        })

    metrics_df = pd.DataFrame(results)
    print("=== Test Set Metrics (GPT2Forecaster) ===")
    print(metrics_df.to_string(index=False))

    return metrics_df, (preds_orig, actuals_orig, stds_orig)

# =============================================
# >>> UNCOMMENT BELOW TO RUN EVALUATION    <<<
# >>> (Requires trained model: best_model)  <<<
# =============================================
metrics_df, _tensors = evaluate_model(
    best_model, test_loader, scaler, TARGET_INDICES, TARGETS,
    HORIZON_LEN, len(ALL_COLS), NUM_TARGETS, n_mc=20
)

In [ ]:
# =============================================================================
# CELL 9 — plot_forecast  (copy-pasted from original — zero changes)
# =============================================================================

def plot_forecast(axes, hist_vals, forecasted, stds, targets, target_labels,
                  target_colors, hist_dates, horizon_actuals=None,
                  history_window=104, prefix=None, country=None):
    """
    axes            : matplotlib Axes array of length n_targets
    hist_vals       : np.ndarray [T, n_targets]            historical (original scale)
    forecasted      : np.ndarray [HORIZON_LEN, n_targets]  model predictions
    stds            : np.ndarray [HORIZON_LEN, n_targets]  MC Dropout std, or None
    hist_dates      : array-like of dates, length T
    horizon_actuals : np.ndarray [HORIZON_LEN, n_targets]  ground truth for window
    history_window  : how many past weeks to show
    """
    hist_vals  = hist_vals[-history_window:]
    hist_dates = pd.to_datetime(hist_dates)[-history_window:]
    fore_vals  = forecasted

    last_date  = hist_dates.iloc[-1] if hasattr(hist_dates, "iloc") else hist_dates[-1]
    fore_dates = pd.date_range(last_date, periods=len(fore_vals) + 1, freq="W")[1:]

    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]

    for i, ax in enumerate(axes):
        color = target_colors[targets[i]]

        ax.plot(hist_dates, hist_vals[:, i],
                color=color, lw=1.2, alpha=0.6, label="Historical")
        ax.plot(fore_dates, fore_vals[:, i],
                color=color, lw=2.0, label="Forecast")
        ax.scatter(fore_dates, fore_vals[:, i], color=color, s=18, zorder=3)

        if stds is not None:
            lower = fore_vals[:, i] - 2 * stds[:, i]
            upper = fore_vals[:, i] + 2 * stds[:, i]
            ax.fill_between(fore_dates, lower, upper,
                            color=color, alpha=0.18, label="95% CI")
            ax.plot(fore_dates, lower, color=color, lw=0.6, alpha=0.4, linestyle=":")
            ax.plot(fore_dates, upper, color=color, lw=0.6, alpha=0.4, linestyle=":")

        if horizon_actuals is not None:
            ax.plot(fore_dates, horizon_actuals[:, i],
                    color="#222222", lw=1.5, linestyle="--", alpha=0.7, label="Actual")
            ax.scatter(fore_dates, horizon_actuals[:, i],
                       color="#222222", s=12, zorder=3, alpha=0.7)

        ax.axvline(last_date, color="#222222", lw=1.2, linestyle="--",
                   alpha=0.5, label="Now")
        ax.axvspan(last_date, fore_dates[-1], color=color, alpha=0.06)

        y_vals = [hist_vals[:, i], fore_vals[:, i]]
        if stds is not None:
            y_vals.append(fore_vals[:, i] - 2 * stds[:, i])
            y_vals.append(fore_vals[:, i] + 2 * stds[:, i])
        if horizon_actuals is not None:
            y_vals.append(horizon_actuals[:, i])
        y_min = min(v.min() for v in y_vals)
        y_max = max(v.max() for v in y_vals)
        ax.set_ylim(y_min * 0.95, y_max * 1.05)

        ax.set_facecolor("#ffffff")

        title = target_labels[targets[i]]
        if i == 0 and (prefix or country):
            title = f"{prefix + ' ' if prefix else ''}" \
                    f"{country + ' — ' if country else ''}{title}"
        ax.set_title(title, fontsize=10, color="#000000", pad=8)

        ax.tick_params(colors="#333333", axis="both")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("#000000")

        if i == 0:
            ax.legend(fontsize=7, facecolor="#ffffff",
                      edgecolor="#000000", labelcolor="#333333")

# TARGET_COLORS = {
#     "respiratory_disease_rate":     "#4fc3f7",
#     "cardio_mortality_rate":        "#81c784",
#     "vector_disease_risk_score":    "#ffb74d",
#     "waterborne_disease_incidents": "#f06292",
#     "heat_related_admissions":      "#ce93d8",
# }

TARGET_COLORS = {
    "respiratory_disease_rate":      "#0288d1",  # Deep Sky Blue
    "cardio_mortality_rate":        "#388e3c",  # Forest Green
    "vector_disease_risk_score":    "#f57c00",  # Vivid Orange
    "waterborne_disease_incidents": "#c2185b",  # Rich Pink/Raspberry
    "heat_related_admissions":      "#7b1fa2",  # Deep Purple
}

In [ ]:
# =============================================================================
# CELL 10 — VISUALIZATION LOOP (fixed)
# =============================================================================

preds_orig, actuals_orig, stds_orig = _tensors

num_of_windows_per_country = preds_orig.shape[0] // NUM_COUNTRIES
print(f"Found a total of ({num_of_windows_per_country}) windows per country")

n_targets = len(TARGETS)

countries = COUNTRIES.copy()
num_countries = len(countries)

# Sorted country codes as groupby sees them — determines block order in preds_orig
sorted_country_codes = sorted(test_combined['country_code'].unique())
sorted_country_index = {code: i for i, code in enumerate(sorted_country_codes)}

fig, axes = plt.subplots(
    num_countries, n_targets,
    figsize=(4.5 * n_targets, 4.2 * num_countries),
    facecolor="#ffffff",
    squeeze=False,
)

for i, country in enumerate(countries):
    # Correct block offset using alphabetical sort index
    sorted_idx = sorted_country_index[country]
    row = (sorted_idx + 1) * num_of_windows_per_country - 1  # last window of this country's block

    # Pull history from test_combined: the SEQ_LEN rows immediately before the forecast horizon
    country_full_df = test_df[test_df["country_code"] == country].sort_values("time_idx")
    history_df      = country_full_df.iloc[-(SEQ_LEN + HORIZON_LEN) : -HORIZON_LEN]

    historical   = history_df[TARGETS].values
    date_values  = history_df["date"].values          # no offset — raw dates
    country_name = country_full_df["country_name"].iloc[-1]

    _forecasted, _ground_truth, _stds = (
        preds_orig[row, ...],
        actuals_orig[row, ...],
        stds_orig[row, ...],
    )

    plot_forecast(
        axes            = axes[i],
        forecasted      = _forecasted,
        horizon_actuals = _ground_truth,
        stds            = _stds,
        hist_vals       = historical,
        targets         = TARGETS,
        target_labels   = TARGET_LABELS,
        target_colors   = TARGET_COLORS,
        hist_dates      = date_values,
        history_window  = 20000,
        prefix          = f"[{i + 1}]",
        country         = country_name,
    )

fig.suptitle(
    "Historical Data and Model Forecasts by Country",
    fontsize=16, color="#eeeeee", y=0.995,
)
fig.patch.set_facecolor("#ffffff")
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

In [ ]:
trainer.save_checkpoint("gpt2-forecaster-epoch=181-step=7098-val_loss=0.201.ckpt")

In [ ]:
metrics_df.to_csv("gpt2-forecaster.csv", index=False)

In [ ]:
fig.savefig("gpt2-forecaster-historical-data-and-model-forecast-all-countries.png", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())

# GPT-2 Decoder Forecasting — Revised Plan

## Architecture Concept (GPT-style for Time Series)

**Training**: Feed full sequence `[0..SEQ_LEN-1]` of concatenated `[features + targets]`. The model predicts the **next token's targets** at every position. Causal mask (built into GPT2) prevents position `i` from seeing future positions. Loss compares `preds[:, t, :]` against `targets[:, t+1, :]` — the classic next-token prediction objective.

**Inference**: Feed the last `SEQ_LEN` context tokens → get 1 prediction → append predicted targets + carry-forward features as a new token → slide window → repeat for `HORIZON_LEN` steps. Uses KV cache for efficiency.

---

## Cell 1 — Constants & Feature Engineering
**Reuse 100% as-is.** Same `FEATURES`, `TARGETS`, `SEQ_LEN=78`, `HORIZON_LEN=12`, same feature engineering, same 70/15/15 split producing `train_df`, `val_df`, `test_df`.

Drop: `KNOWN_REALS`, `UNKNOWN_REALS` — TFT-specific concepts, irrelevant here.

---

## Cell 2 — Dataset & DataLoaders

Replace `TimeSeriesDataSet` with a plain `torch.utils.data.Dataset`.

**What each sample looks like:**
```
input_seq  : X[k : k + SEQ_LEN]        shape [78, 28]   (23 features + 5 targets)
target_seq : X[k+1 : k + SEQ_LEN + 1]  shape [78,  5]   (targets only, shifted +1)
```

> **Key**: The shift happens in the dataset, NOT in the loss. `target_seq[t]` is the ground truth for what position `t` in `input_seq` should predict.

**Normalization:** Fit `StandardScaler` on `train_df[ALL_COLS]` only, apply to all splits. Store target column indices separately for `inverse_transform` at eval time.

**DataLoaders:** `DataLoader(dataset, batch_size=64, shuffle=True)` for train, `shuffle=False` for val/test. No `collate_fn` needed.

---

## Cell 3 — Model Definition

Three components in one `LightningModule` (`GPT2Forecaster`):

```
Component 1 — Input Projection
    nn.Linear(28, d_model)
    Maps each token [features + targets] → d_model embedding space

Component 2 — GPT2 Backbone
    GPT2Config(
        n_embd=d_model, n_layer=4, n_head=4,
        n_positions=SEQ_LEN,
        resid_pdrop=0.1, attn_pdrop=0.1,
        use_cache=True
    )
    GPT2Model(config)
    → Causal mask is built-in, no manual masking needed
    → past_key_values (KV cache) handled automatically at inference

Component 3 — Output Head
    nn.Linear(d_model, 5)
    Maps each hidden state → 5 target predictions
```

**Forward pass (training):**
```
input_seq [B, 78, 28]
    → input_proj → [B, 78, d_model]
    → GPT2Model(inputs_embeds=...) → hidden [B, 78, d_model]
    → output_head → preds [B, 78, 5]

loss = HuberLoss(preds, target_seq)
    ↑ No additional shift needed — target_seq is already shifted +1 in the dataset
```

**LightningModule hooks:**
```
training_step       → forward + loss
validation_step     → forward + loss + log val_loss
configure_optimizers → AdamW(lr=1e-3, weight_decay=1e-2) + ReduceLROnPlateau(patience=5)
```

---

## Cell 4 — Training

```python
def train_model(model, train_loader, val_loader, max_epochs=100, patience=25):
    early_stop = EarlyStopping(monitor="val_loss", patience=patience, mode="min")
    lr_monitor = LearningRateMonitor(logging_interval="epoch")
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        gradient_clip_val=0.1,  # tighter clipping for transformer stability
        callbacks=[early_stop, lr_monitor],
        log_every_n_steps=5,
    )
    trainer.fit(model, train_loader, val_loader)
    best = GPT2Forecaster.load_from_checkpoint(
        trainer.checkpoint_callback.best_model_path
    )
    return best, trainer
```

> Training call is commented out for manual execution.

---

## Cell 5 — Evaluation

Keep existing `evaluate_tft` structure with two changes:

**Change 1 — Autoregressive inference** replaces `model.predict(test_loader)`:
```
For each test sample:
    context = last SEQ_LEN tokens from history
    for step in range(HORIZON_LEN):
        pred = model.forward(context)    # uses KV cache
        next_token = [carry-forward features, predicted targets]
        context = concat(context, next_token)[-SEQ_LEN:]   # slide window
    Collect 12 predictions → [B, 12, 5]
```

**Change 2 — Inverse transform** before metrics:
```
predictions and actuals are in normalized space
→ apply target_scaler.inverse_transform() before MAE/RMSE/R²
```

**Uncertainty bands**: Replace quantile bands (q10/q90) with ±1 std from MC dropout (enable dropout at inference, run N forward passes, compute mean ± std).

Everything else — metrics loop (MAE, RMSE, R², Peak MAE, Dir Acc%), plot layout, dark theme — stays identical.

---

## Notes
- Use built-in HuggingFace `GPT2Model` — no need to reimplement attention or masking
- `d_model` can be tuned (64 is a good starting point for this dataset size)
- The shift-by-1 happens **once** in the dataset, not in the loss function
- At inference, features for future steps must be carried forward (calendar features like `month_sin/cos` can be computed; other features use last known value)